# 1. Giới thiệu về dữ liệu

Bộ dữ liệu **Pima Indians Diabetes Database** gồm **768 mẫu**, mỗi mẫu đại diện cho **một bệnh nhân nữ người da đỏ Pima**, với **8 thuộc tính lâm sàng** như:

- **Pregnancies:** Số lần mang thai  
- **Glucose:** Mức glucose huyết tương  
- **BloodPressure:** Huyết áp tâm trương  
- **SkinThickness:** Độ dày lớp da  
- **Insulin:** Nồng độ insulin  
- **BMI:** Chỉ số khối cơ thể  
- **DiabetesPedigreeFunction:** Chỉ số di truyền tiểu đường  
- **Age:** Tuổi  

Các thuộc tính này đều là **dữ liệu liên tục**, trong khi thuật toán **ID3** chỉ hoạt động tốt trên **dữ liệu rời rạc (categorical)**.  
Do đó, cần thực hiện **Binning (phân nhóm)** để chuyển đổi dữ liệu liên tục thành các mức độ (thấp, trung bình, cao...) dựa trên **chuẩn y tế**.

---

### Mục đích của Binning

- Giảm nhiễu và giúp mô hình dễ giải thích hơn  
  (ví dụ: “Glucose cao” dễ hiểu hơn giá trị 185).  
- Cho phép ID3 chọn thuộc tính bằng **Information Gain** chính xác hơn.  
- Tăng khả năng **diễn giải kết quả** trong bối cảnh y tế.

---

## BINNING DỰA TRÊN TIÊU CHUẨN Y TẾ

**Pregnancies:** Thấp (0–2), Trung bình (3–6), Cao (>6)  
**Glucose:** Bình thường (<140), Tiền tiểu đường (140–199), Tiểu đường (≥200)  
**BloodPressure:** Bình thường (<80), Cao độ 1 (80–89), Cao độ 2 (≥90)  
**SkinThickness:** Thấp (<20mm), Trung bình (20–30mm), Cao (>30mm)  
**Insulin:** Bình thường (<100), Cao (100–200), Rất cao (>200)  
**BMI:** Bình thường (<25), Thừa cân (25–30), Béo phì (>30)  
**DiabetesPedigreeFunction:** Thấp (<0.3), Trung bình (0.3–0.6), Cao (>0.6)  
**Age:** Trẻ (<30), Trung niên (30–50), Lớn tuổi (>50)

---

#  2. Mô tả chi tiết thuật toán ID3

Thuật toán **ID3 (Iterative Dichotomiser 3)** là một **phương pháp học có giám sát (supervised learning)** dùng để **xây dựng cây quyết định (Decision Tree)** dựa trên chỉ số **Information Gain**, thể hiện mức độ giảm **Entropy (độ bất định)** khi chia tách dữ liệu theo một thuộc tính.

###  Quy trình huấn luyện

1. **Chọn thuộc tính** có **Information Gain lớn nhất** làm nút gốc.  
2. **Phân tách dữ liệu** theo các giá trị của thuộc tính đó.  
3. **Lặp lại** quá trình trên cho từng nhánh cho đến khi:  
   - Tất cả các mẫu trong nhánh thuộc cùng một lớp, **hoặc**  
   - Không còn thuộc tính nào để chia tách.  

Mỗi **nút lá (leaf node)** của cây thể hiện kết quả dự đoán:  
- `0`: Không tiểu đường  
- `1`: Có tiểu đường

---

# 3. Lý do chọn thuật toán ID3

- **Phù hợp** với dữ liệu đã được **rời rạc hóa (sau binning)** như Pima.  
- **Dễ giải thích**: mỗi nhánh của cây thể hiện một **chuỗi điều kiện y tế** rõ ràng  
  *(ví dụ: Glucose cao → BMI cao → Nguy cơ tiểu đường).*  
- **Hữu ích trong báo cáo y tế** hoặc **công cụ hỗ trợ bác sĩ**,  
  vì có thể **hiển thị logic chẩn đoán minh bạch**.  
- **Phù hợp với mục tiêu diễn giải** (interpretability) hơn là dự đoán thuần túy.


 **Ý nghĩa trong bối cảnh y tế**

Giúp xác định thuộc tính nào quan trọng nhất trong việc chẩn đoán tiểu đường.  
Ví dụ, nếu **Glucose** có Information Gain lớn nhất, điều đó cho thấy mức đường huyết là yếu tố quyết định chính.  

Nhờ đó, mô hình không chỉ dự đoán mà còn **giải thích được lý do** của từng quyết định, hỗ trợ phân tích y khoa minh bạch.



In [2]:
############################################################
# THUẬT TOÁN ID3 (DECISION TREE)
############################################################

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings('ignore')

# ===========================
# ĐỌC DỮ LIỆU
# ===========================
df1 = pd.read_csv("diabetes_dataset1_mean.csv")
df2 = pd.read_csv("diabetes_dataset2_median.csv")

# ===========================
# HÀM ĐÁNH GIÁ MÔ HÌNH
# ===========================
def evaluate_model(y_true, y_pred, model_name, dataset_name):
    print(f"\n{'='*60}")
    print(f"{model_name} - {dataset_name}")
    print(f"{'='*60}")

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    print(f"- Accuracy:  {accuracy:.4f}")
    print(f"- Precision: {precision:.4f}")
    print(f"- Recall:    {recall:.4f}")
    print(f"- F1-Score:  {f1:.4f}")

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=['Không bị', 'Bị']))

    # return accuracy, precision, recall, f1

# ===========================
# HÀM BINNING DỮ LIỆU (ID3)
# ===========================
def prepare_data_for_id3(df):
    df_binned = df.copy()
    df_binned['Pregnancies_binned'] = pd.cut(df['Pregnancies'], bins=[-0.1, 2, 6, 20], labels=['Thấp', 'TB', 'Cao'])
    df_binned['Glucose_binned'] = pd.cut(df['Glucose'], bins=[0, 140, 200, 300], labels=['BT', 'Tiền', 'Tiểu'])
    df_binned['BloodPressure_binned'] = pd.cut(df['BloodPressure'], bins=[0, 80, 90, 150], labels=['BT', 'Cao1', 'Cao2'])
    df_binned['SkinThickness_binned'] = pd.cut(df['SkinThickness'], bins=[0, 20, 30, 100], labels=['Thấp', 'TB', 'Cao'])
    df_binned['Insulin_binned'] = pd.cut(df['Insulin'], bins=[0, 100, 200, 900], labels=['BT', 'Cao', 'Rất cao'])
    df_binned['BMI_binned'] = pd.cut(df['BMI'], bins=[0, 25, 30, 70], labels=['BT', 'Thừa', 'Béo'])
    df_binned['DiabetesPedigreeFunction_binned'] = pd.cut(df['DiabetesPedigreeFunction'], bins=[0, 0.3, 0.6, 3], labels=['Thấp', 'TB', 'Cao'])
    df_binned['Age_binned'] = pd.cut(df['Age'], bins=[0, 30, 50, 100], labels=['Trẻ', 'Trung', 'Già'])
    return df_binned

# ===========================
# TRAIN & TEST ID3
# ===========================
print("\n--- ID3 với Dataset 1 (Mean) ---")
df1_binned = prepare_data_for_id3(df1)
le = LabelEncoder()
X1 = df1_binned[[c for c in df1_binned.columns if c.endswith('_binned')]].apply(le.fit_transform)
y1 = df1_binned['Outcome']
X1_train, X1_test, y1_train, y1_test = train_test_split(X1, y1, test_size=0.2, random_state=42)
model1 = DecisionTreeClassifier(criterion='entropy', max_depth=5, random_state=42)
model1.fit(X1_train, y1_train)
y1_pred = model1.predict(X1_test)
evaluate_model(y1_test, y1_pred, "ID3", "Dataset 1 (Mean)")

print("\n--- ID3 với Dataset 2 (Median) ---")
df2_binned = prepare_data_for_id3(df2)
X2 = df2_binned[[c for c in df2_binned.columns if c.endswith('_binned')]].apply(le.fit_transform)
y2 = df2_binned['Outcome']
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=42)
model2 = DecisionTreeClassifier(criterion='entropy', max_depth=5, random_state=42)
model2.fit(X2_train, y2_train)
y2_pred = model2.predict(X2_test)
evaluate_model(y2_test, y2_pred, "ID3", "Dataset 2 (Median)")



--- ID3 với Dataset 1 (Mean) ---

ID3 - Dataset 1 (Mean)
- Accuracy:  0.7338
- Precision: 0.6750
- Recall:    0.4909
- F1-Score:  0.5684

Confusion Matrix:
[[86 13]
 [28 27]]

Classification Report:
              precision    recall  f1-score   support

    Không bị       0.75      0.87      0.81        99
          Bị       0.68      0.49      0.57        55

    accuracy                           0.73       154
   macro avg       0.71      0.68      0.69       154
weighted avg       0.73      0.73      0.72       154


--- ID3 với Dataset 2 (Median) ---

ID3 - Dataset 2 (Median)
- Accuracy:  0.7338
- Precision: 0.6750
- Recall:    0.4909
- F1-Score:  0.5684

Confusion Matrix:
[[86 13]
 [28 27]]

Classification Report:
              precision    recall  f1-score   support

    Không bị       0.75      0.87      0.81        99
          Bị       0.68      0.49      0.57        55

    accuracy                           0.73       154
   macro avg       0.71      0.68      0.69       1

# 🧩 KẾT QUẢ VÀ ĐÁNH GIÁ MÔ HÌNH ID3 (Decision Tree)

## 📊 Kết quả huấn luyện

### **Dataset 1 (Mean)**
- **Accuracy:** 0.7338  
- **Precision:** 0.6750  
- **Recall:** 0.4909  
- **F1-Score:** 0.5684  

**Confusion Matrix:**

|                  | Dự đoán Không bị | Dự đoán Bị |
|------------------|------------------|-------------|
| **Thực tế Không bị** | 86 | 13 |
| **Thực tế Bị**        | 28 | 27 |

**Nhận xét:**
- Mô hình dự đoán đúng **73.38%** tổng số mẫu thử.  
- Độ **chính xác (Precision)** đạt **67.5%**, tức là trong các mẫu được dự đoán là “Bị tiểu đường”, có khoảng 2/3 là đúng.  
- **Recall (49.09%)** cho thấy mô hình bỏ sót khá nhiều ca bệnh thật.  
- **F1-Score (0.5684)** thể hiện sự cân bằng trung bình giữa precision và recall.

---

### **Dataset 2 (Median)**
- **Accuracy:** 0.7338  
- **Precision:** 0.6750  
- **Recall:** 0.4909  
- **F1-Score:** 0.5684  

**Confusion Matrix:**

|                  | Dự đoán Không bị | Dự đoán Bị |
|------------------|------------------|-------------|
| **Thực tế Không bị** | 86 | 13 |
| **Thực tế Bị**        | 28 | 27 |

**Nhận xét:**  
Kết quả tương tự Dataset 1 → Việc thay giá trị thiếu bằng **mean** hay **median** không ảnh hưởng đáng kể đến hiệu quả mô hình ID3.

---

## 🧠 Đánh giá và Nhận xét

### 🔹 Lý do chọn thuật toán
- ID3 là **thuật toán cây quyết định** đơn giản, trực quan và dễ hiểu.  
- Phù hợp khi dữ liệu đã được **rời rạc hóa (binning)**.  
- Giúp nhận biết yếu tố nào ảnh hưởng nhiều nhất đến khả năng mắc bệnh tiểu đường.

### 🔹 Ưu điểm
- **Dễ giải thích**: có thể trực quan hóa cây để xem đặc trưng quan trọng.  
- Không yêu cầu dữ liệu tuân theo phân phối chuẩn.  
- Hoạt động tốt với dữ liệu rời rạc sau binning.

### 🔹 Hạn chế
- **Hiệu suất dự đoán trung bình (~73%)**.  
- **Recall thấp (~49%)**, mô hình bỏ sót nhiều ca bệnh thật.  
- **Dễ bị overfitting** nếu không giới hạn độ sâu hoặc không cắt tỉa (pruning).

### 🔹 Tổng kết
- Mô hình ID3 cho kết quả **ổn định và dễ diễn giải**, nhưng chưa đạt độ chính xác cao.  
- Thích hợp để **phân tích mối quan hệ giữa các thuộc tính và kết quả**, hơn là dùng để dự đoán thuần túy.  
- Cần so sánh thêm với các mô hình khác như **GaussianNB** và **KNN** để chọn giải pháp tối ưu hơn.
